# peft-telugu-bench on a Kaggle GPU

1. **Settings → Accelerator → GPU T4 x2** (only one GPU is used, so results match the Colab LoRA run), and **Settings → Internet → On**.
2. **Save Version → Save & Run All (Commit)**. It runs in the background (tab can be closed), DoRA then QLoRA, about 5 hours.
3. When it finishes, open the version's **Output** tab: `results/<method>-r16/metrics.json`, and the last cell prints both.

In [ ]:
import os
# Two T4s would make the trainer split each batch across both GPUs, doubling the effective batch size
# and making the results incomparable with the single-T4 LoRA run.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
%cd /kaggle/working
!git clone https://github.com/amarthya-14/peft-telugu-bench.git 2>/dev/null || git -C peft-telugu-bench pull
%cd /kaggle/working/peft-telugu-bench
!git log --oneline -1

In [ ]:
!pip install -q 'transformers>=5.20.2' 'peft>=0.21.2' 'datasets>=5.1.0' 'accelerate>=1.15.0' bitsandbytes sacrebleu
# peft 0.21.2 rejects old preinstalled torchao versions even though we never use it.
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
METHODS = ["dora", "qlora"]
for method in METHODS:
    !python train.py --method {method} --out_dir /kaggle/working/results

In [ ]:
!cat /kaggle/working/results/*/metrics.json